# 작은 mask-diffusion LM 직접 만들기 — 가변 마스킹과 병렬 denoise

Phase 5 의 첫 챕터. Ch 24-31 까지 다룬 **GPT (decoder, autoregressive, 왼→오 순차 생성)** 패러다임에서, 이번엔 **Diffusion LM (encoder/bidirectional, masked-denoise, 문장 전체를 병렬로 생성)** 패러다임으로 전환합니다.

학습이 끝나면 전부 `[MASK]` 인 빈 캔버스에서 시작해, 왼→오가 아니라 **병렬로** 토큰을 채워가며 영어 동화를 만들어냅니다.

> *"Once upon a time, there was a boy named Timmy. He had found a big toy ball in the park. He went to a big house with his toys. He liked to play with him."*

핵심 한 줄: **BERT MLM (Ch 20-23) 의 *고정 15% 마스킹* 을 *0-100% 가변 마스킹* 으로 일반화하고, 한 번에 복원하는 대신 *여러 번 반복 denoise* 하면 그게 generation 입니다.** Ch 1 부터 추적해 온 *마스킹 + 토크나이저* 시각이 여기서 클라이맥스에 도달합니다 — 가려서 맞히던 BERT 가, 가리는 비율을 끝까지 밀어붙이면 *무에서 문장을 만들어내는 생성 모델* 이 됩니다.

작은 BERT-style 모델을 *random init* 으로 from scratch 띄우고, **TinyStories** 로 *가변 마스킹 denoising* 목표로 학습 → reverse process (전부 `[MASK]` 에서 시작해 반복 denoise) 로 텍스트를 *왼→오가 아닌 병렬* 로 생성하는 과정을 직접 눈으로 봅니다.

**환경**: Google Colab **T4 GPU 필수**. **예상 소요**: 약 25분 (데이터·토큰화 약 5분 + 학습 약 20분 + 생성·궤적).

---

## 학습 흐름

1. 📊 **변화 추적표 + Phase 전환 도입부** — Autoregressive (GPT) → Diffusion 큰 그림
2. 🔄 **변경점** — 생성 방식 (순차 → 병렬 denoise), attention (causal → bidirectional), 마스킹 (고정 15% → 가변 0-100%)
3. 📐 **Loss** — masked-diffusion denoising loss. MLM 의 CE 를 *가변 마스킹 비율 t* 로 일반화 + `1/t` 재가중
4. 💡 **마스킹 thread 클라이맥스** — BERT 의 *고정 15%* vs diffusion 의 *가변 0-100%*. 같은 `-100` 트릭
5. 🔤 **토크나이저 노트** — 작은 모델엔 작은 vocab (ByteLevel BPE 2048 직접 학습 + `[MASK]`)
6. 🚀 **실습**: TinyStories → 작은 BERT-style 모델 → 가변 마스킹 denoising 학습
7. 🔬 **Reverse process generation** — 전부 `[MASK]` 에서 반복 denoise. 마스크가 *병렬로* 단어로 채워지는 궤적 직접 관찰
8. 🛠️ **변형**: denoise step 수 비교 (1 / 4 / 16 / 32), 조건부 생성 (prompt 고정)
9. ⚖️ **AR vs Diffusion 비교** — Ch 24 (GPT) 와 나란히. Ch 33 (샘플러)·Ch 34 (한국어) 예고
10. 📦 **등장 라이브러리** / 🎯 **체크포인트** / ❓ **FAQ** (답변 포함)

---

> 📒 **사전 학습 자료**: Ch 20-23 (BERT MLM 사전학습 — 고정 15% 마스킹), Ch 24 (GPT from scratch — autoregressive generation). 본 챕터는 *둘을 잇습니다* — BERT 의 *마스킹-복원* 메커니즘을 Ch 24 의 *generation* 목적에 다시 씁니다. 다른 점은 *마스킹 비율을 0-100% 로 일반화* 하고 *복원을 여러 번 반복* 한다는 것뿐.

## 📊 변화 추적표

| Ch | 모델 | 토크나이저 | 데이터 | Output Head | 생성/학습 방식 | Loss |
|---|---|---|---|---|---|---|
| 20 | 작은 BERT (영어, scratch) | `bert-base-uncased` (가져옴) | Wikitext-103 | MLM head | 고정 15% 마스킹-복원 | `CrossEntropyLoss` (masked 15%) |
| 24 | 작은 GPT2 (직접, scratch) | BPE (직접 학습) | TinyStories | `Linear(H, V)` | autoregressive (왼→오 순차) | `CrossEntropyLoss` (next-token) |
| 31 | SFT base + GRPO | BBPE | verifiable-reward | `Linear(H, V)` + group adv. | autoregressive + RL | `GRPO loss` |
| **32 ← 여기** | **작은 BERT-style (직접, scratch)** | **ByteLevel BPE 2048 (직접 학습 + `[MASK]`)** | **TinyStories** | **`Linear(H, V)`** | **parallel denoise (가변 마스킹 + 반복 복원)** | **masked-diffusion denoising loss (`1/t` 재가중)** |
| 33 (다음) | MDLM (170M) / DiffuGPT (124M) 사전학습 | (각 모델 토크나이저) | 영어 사전학습 추론 시연 | `Linear(H, V)` | parallel denoise (추론만) | — |

전체 챕터 표는 [루트 README](https://github.com/yoon-gu/neuqes-101#챕터별-변화추적표) 를 참고하세요.

---

## Phase 전환 — Autoregressive (GPT) → Diffusion LM

Ch 24-31 의 GPT 챕터들이 *decoder + next-token 예측 + 왼→오 순차 생성* 패러다임이라면, Phase 5 (Ch 32-34) 는 *encoder/bidirectional + masked-denoise + 문장 전체 병렬 생성* 패러다임입니다. 본 챕터가 그 출발점.

| 축 | Phase 4 (GPT, Ch 24-31) | **Phase 5 (Diffusion, Ch 32-34)** |
|---|---|---|
| attention | Causal (과거만 봄) | **Bidirectional (양방향 다 봄)** |
| 학습 목표 | next-token 예측 | **가변 마스킹 denoising** |
| 생성 순서 | 왼→오 *한 토큰씩 순차* | **문장 전체를 *동시에* 여러 번 denoise** |
| 생성 step 수 | 토큰 수 = step 수 (길면 느림) | **step 수를 *자유롭게 조절* (4 / 16 / 32 ...)** |
| 출발 상태 | prompt 토큰들 | **전부 `[MASK]` (무에서 시작)** |
| 본체 계보 | GPT (Ch 24) | **BERT (Ch 20)** — MLM 을 일반화 |

> **핵심 직관**: GPT 가 *왼쪽부터 한 글자씩 받아쓰기* 라면, diffusion 은 *흐릿한 전체 그림을 여러 번 선명하게 다듬기* 입니다. 이미지 생성에서 노이즈를 점점 걷어내듯, 텍스트에서는 `[MASK]` 를 점점 진짜 단어로 바꿔 갑니다. 본 챕터는 그 메커니즘을 *작은 모델로 직접 구현* 해 봅니다. Ch 33 (MDLM 170M / DiffuGPT 124M 사전학습) 이 *같은 원리의, 충분한 규모로 학습된 실전 모델* 입니다.

## 🔄 변경점 (Diff from Ch 31)

| 축 | Ch 24-31 (GPT, autoregressive) | Ch 32 (Diffusion LM) |
|---|---|---|
| **생성 방식** | next-token, 왼→오 *순차* | **masked-denoise, 문장 전체 *병렬*** ← *Phase 전환의 핵심* |
| attention | Causal (`GPT2LMHeadModel`) | **Bidirectional (`BertForMaskedLM` 계열)** |
| 학습 목표 | `CrossEntropyLoss` (next-token, 거의 모든 자리) | **masked-diffusion loss (가변 비율 `t` 마스킹 + `1/t` 재가중)** |
| 마스킹 | 없음 (causal mask 가 미래 차단) | **입력 토큰을 `t` 비율로 `[MASK]` 치환** |
| 토크나이저 | BPE (GPT 계열) | **ByteLevel BPE 2048 직접 학습 + `[MASK]`** |
| 생성 출발 | prompt 토큰 | **전부 `[MASK]` 인 시퀀스** |
| 생성 step | 토큰 길이 만큼 | **임의 step 수 (속도-품질 trade-off 조절 가능)** |

> **변경점이 한꺼번에 많은 이유** — Phase 가 바뀌는 *전환 챕터* 라 *축 자체* 가 새로 정의됩니다. 하지만 본질은 *Ch 20 의 BERT MLM 을 재활용* 한 것 — *bidirectional + 마스킹-복원* 은 이미 다 배운 메커니즘이고, *마스킹 비율을 가변* 으로 만들고 *복원을 반복* 한 것만 새롭습니다. 다음 두 챕터는 다시 *한 가지씩* 만 바뀝니다 — **Ch 33: 샘플러를 바꿔 생성 품질을 끌어올리고, Ch 34: 한국어로 확장**합니다.

## 📐 Loss — masked-diffusion denoising loss

BERT MLM 의 CrossEntropyLoss 와 *뼈대는 같습니다* — 가려진 자리의 정답 토큰을 맞히는 CE. 다른 점은 두 가지:

1. 마스킹 비율이 *고정 15%* 가 아니라 *매 샘플마다 $t \sim U(0, 1)$ 로 뽑은 가변 비율*
2. 비율 $t$ 만큼 가렸으니, loss 를 *$1/t$ 로 재가중* 해 *어떤 마스킹 비율이든 공정하게* 평균

### 수식

깨끗한 토큰 시퀀스 $x_0 = (x_1, \dots, x_L)$ 에 대해, 비율 $t$ 를 뽑고 각 토큰을 *독립적으로 확률 $t$* 로 `[MASK]` 치환해 $x_t$ 를 만듭니다. 모델은 $x_t$ 전체를 보고 *가려진 자리* 의 원본 토큰을 예측:

$$L = \mathbb{E}_{t \sim U(0,1)} \left[ \frac{1}{t} \cdot \frac{1}{L} \sum_{i:\, x_t^{(i)} = \texttt{[MASK]}} -\log P_\theta\!\left(x_0^{(i)} \mid x_t\right) \right]$$

- $\sum_{i:\, x_t^{(i)}=\texttt{[MASK]}}$: *가려진 자리에서만* loss 계산 (Ch 20-23 의 `-100` 트릭과 동일)
- $1/t$ 재가중: $t$ 가 작으면 (조금 가림) 가려진 토큰이 적으니 합이 작아지는데, $1/t$ 로 곱해 *마스킹 비율에 무관* 하게 스케일을 맞춤. 이 재가중 덕분에 *학습 목표가 log-likelihood 의 upper bound* 가 됩니다 (LLaDA / MDLM 의 핵심 항)
- $t \sim U(0,1)$: 매 step 마다 *다른 난이도* 의 복원 문제를 풀게 함 — 5% 만 가린 쉬운 문제부터 95% 가린 거의-무에서-생성 문제까지

### 숫자로 감 잡기 (vocab = 2,048)

random init 직후 모델은 가려진 자리를 *균등 추측* → 정답 확률 $1/2048$, 토큰당 $-\log p \approx \ln(2048) = 7.62$.

| 마스킹 비율 $t$ | 가린 토큰 수 (L=128) | 가린 자리 합 ($\approx t L \times 7.62$) | $\times \frac{1}{t}\frac{1}{L}$ 후 | 해석 |
|---|---|---|---|---|
| 0.10 | 약 13 | 약 99 | **7.62** | 조금 가린 쉬운 복원 |
| 0.50 | 약 64 | 약 488 | **7.62** | 절반 가림 |
| 0.90 | 약 115 | 약 877 | **7.62** | 거의 무에서 생성 |

**관전 포인트**:
- `1/t` 재가중 덕분에 *어떤 t 든 baseline loss 가 똑같이 `ln(vocab) ≈ 7.62`* 로 정렬됩니다. 직접 학습한 BPE 2048 의 random baseline `ln(2048) ≈ 7.62` 와 같은 값 — *마스킹 비율만 일반화* 했지 loss 의 척도는 그대로입니다.
- 학습 첫 step loss 가 약 7.6 부근에서 시작해 빠르게 떨어지면 정상. 목표는 약 3.7 부근 (작은 모델 + TinyStories).
- $t$ 가 1 에 가까울수록 (거의 다 가림) *문맥 정보가 거의 없어* 복원이 어려움 → diffusion 생성이 *여러 step 에 나눠* 조금씩 푸는 이유.

## 💡 마스킹 thread 클라이맥스 — *고정 15%* (BERT) → *가변 0-100%* (diffusion)

Ch 1 부터 *토크나이저와 마스킹* 을 일관되게 추적해 왔습니다. 그 흐름이 여기서 정점에 닿습니다.

| 단계 | 챕터 | 마스킹 비율 | 복원 횟수 | 용도 |
|---|---|---|---|---|
| MLM 사전학습 | Ch 20 (영어), Ch 22 (한국어) | **고정 15%** | 1회 | 표현 학습 (downstream fine-tune 용) |
| GPT CausalLM | Ch 24-31 | 마스킹 없음 (causal mask) | — | autoregressive 생성 |
| **Mask-diffusion** | **Ch 32 (본 챕터)** | **가변 $t \sim U(0,1)$** | **반복 (4-32 step)** | **병렬 생성** |

핵심은 **셋이 모두 같은 `labels = -100` 트릭** 을 쓴다는 점입니다 — *가려진 자리만* loss 계산, 나머지는 `-100` 으로 무시. Ch 20 에서 손에 익힌 그 패턴이 그대로 재등장합니다.

> **"가린다" 의 의미가 바뀝니다.** BERT 에서 마스킹은 *표현을 배우기 위한 수단* (15% 만 살짝 가려 문맥으로 복원). Diffusion 에서 마스킹은 *생성 그 자체* — 100% 가린 `[MASK]` 시퀀스에서 출발해 한 step 씩 단어를 채우면, 그게 *무에서 문장을 만들어내는 것* 입니다. **같은 메커니즘, 다른 목적.** 가리는 비율을 끝까지 밀어붙였더니 *복원이 생성이 되었습니다.*

이 챕터에서 학습 collator 는 매 배치마다 $t$ 를 새로 뽑아 *가변 비율* 로 가립니다. Ch 20 의 `DataCollatorForLanguageModeling(mlm=True, mlm_probability=0.15)` 가 *고정 15%* 였다면, 여기서는 *직접 만든 가변 collator* 가 *0-100% 를 매번 다르게* 가립니다.

## 🔤 토크나이저 노트 — 작은 모델엔 작은 vocab (BPE 2048 + `[MASK]`)

Diffusion LM 의 주인공 토큰은 **`[MASK]`** 입니다. 전부 `[MASK]` 인 빈 캔버스에서 시작해 토큰을 채우는 게 곧 생성이니까요.

그런데 작은 from-scratch 모델에는 **작은 vocab** 이 중요합니다. `bert-base-uncased` 의 WordPiece 는 vocab 이 30,522 개라, hidden 256 짜리 작은 모델에 그대로 붙이면 임베딩 테이블이 파라미터의 대부분을 잡아먹어 정작 문맥을 배우는 본체에 쓸 용량이 없습니다. 그래서 이 챕터는 Ch 19·24 처럼 **TinyStories 코퍼스에 ByteLevel BPE 2048 을 직접 학습** 하고, 거기에 `[PAD]`·`[UNK]`·`[MASK]` 특수 토큰을 더해 씁니다.

| 토크나이저 | vocab | `[MASK]` | 본 챕터 적합성 |
|---|---|---|---|
| WordPiece (`bert-base-uncased`) | 30,522 | 내장 | 작은 모델엔 임베딩 과대 |
| BPE (GPT-2) | 50,257 | 없음 | 별도 추가 필요 |
| **ByteLevel BPE (직접 학습)** | **2048** | **추가** | **작은 모델에 딱 — 본 챕터** |

`[MASK]` 토큰이 *forward process (가리기)* 와 *reverse process (복원/생성)* 양쪽의 핵심입니다.

> Ch 1 부터 추적한 *토크나이저 시각* 의 클라이맥스 — *같은 문장이 어떻게 토큰화되는가* 를 넘어, 이제 *`[MASK]` 토큰 자체가 생성의 캔버스* 가 됩니다.

## 🛠️ 환경 셋업

In [ ]:
%pip install -q -U transformers tokenizers datasets accelerate

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import math
import os
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# device 자동 감지 - Colab T4 / 로컬 MPS / CPU 모두 지원
if torch.cuda.is_available():
    device = torch.device("cuda")
    device_name = torch.cuda.get_device_name(0)
    vram_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"device     : cuda  ({device_name})")
    print(f"VRAM total : {vram_gib:.2f} GiB")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("device     : mps  (Apple Silicon)")
else:
    device = torch.device("cpu")
    print("device     : cpu  (training will be very slow - Colab T4 recommended)")

print(f"torch      : {torch.__version__}")

# 재현성
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# fp16 은 CUDA 에서만 (MPS 는 미지원, CPU 는 의미 없음)
USE_FP16 = (device.type == "cuda")
print(f"use fp16   : {USE_FP16}")

# matplotlib 한글 폰트 (Colab — NanumGothic). plot 의 한국어가 □ 로 깨지지 않게.
import matplotlib.pyplot as plt, matplotlib.font_manager as fm, subprocess, os
_fp = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_fp):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True)
fm.fontManager.addfont(_fp)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False

## 1. TinyStories 데이터 로드

Ch 24 (GPT) 와 *완전히 같은 데이터* — `roneneldan/TinyStories` (Eldan & Li 2023, arXiv:2305.07759). GPT-3.5 / GPT-4 가 *4세 어린이 어휘* 로 생성한 짧은 영어 동화. 어휘·문법이 단순해 작은 모델로도 의미 있는 생성이 가능합니다.

*데이터를 Ch 24 와 동일* 하게 둔 이유: 나중에 *같은 데이터에서 AR (Ch 24) vs Diffusion (본 챕터) 생성 방식만 다른* 비교를 하기 위함입니다.

학습 split 의 처음 **30,000 stories** 만 사용 (T4 30분 룰 안).

In [ ]:
from datasets import load_dataset

N_TRAIN = 100_000      # 더 길게 돌리려면 키우세요 (full 은 약 2.1M stories)
N_VAL   = 500

raw_train = load_dataset("roneneldan/TinyStories", split=f"train[:{N_TRAIN}]")
raw_val   = load_dataset("roneneldan/TinyStories", split=f"validation[:{N_VAL}]")
print("train:", raw_train)
print("val  :", raw_val)
print("\n=== sample story ===")
print(raw_train[0]["text"][:400])

## 2. ByteLevel BPE 2048 직접 학습 + `[MASK]` 추가

Ch 19·24 처럼 TinyStories 코퍼스에 ByteLevel BPE 를 vocab 2,048 으로 직접 학습하고, `[PAD]`·`[UNK]`·`[MASK]` 특수 토큰을 더해 씁니다. 핵심은 `[MASK]` 토큰의 존재 (직접 학습이라 `special_tokens` 로 명시 추가).

In [ ]:
# 작은 모델엔 작은 vocab — TinyStories 에 BPE 2048 직접 학습 + [MASK] 추가
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast
VOCAB = 2048
def corpus_iter(bs=1000):
    for i in range(0, len(raw_train), bs):
        yield raw_train[i:i+bs]["text"]
_tk = Tokenizer(models.BPE(unk_token="[UNK]"))
_tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
_tk.decoder = decoders.ByteLevel()
_tk.train_from_iterator(corpus_iter(), trainer=trainers.BpeTrainer(
    vocab_size=VOCAB, special_tokens=["[PAD]", "[UNK]", "[MASK]"]))
tokenizer = PreTrainedTokenizerFast(tokenizer_object=_tk, pad_token="[PAD]",
                                    unk_token="[UNK]", mask_token="[MASK]")
print(f"vocab_size : {tokenizer.vocab_size}")
print(f"[MASK]     : '{tokenizer.mask_token}'  id={tokenizer.mask_token_id}")

# [MASK] 가 섞인 시퀀스 = diffusion 의 노이즈. 일부를 가려본다
import random as _r; _r.seed(0)
sample = "Once upon a time, a little rabbit went to the forest."
enc = tokenizer(sample, add_special_tokens=False)["input_ids"]
md = enc[:]
for i in sorted(_r.sample(range(len(enc)), max(1, len(enc)//3))):
    md[i] = tokenizer.mask_token_id
print("\noriginal :", sample)
print("masked   :", tokenizer.decode(md))

**관전 포인트** — `[MASK]` 가 섞인 시퀀스가 바로 diffusion 의 *중간 상태* $x_t$ 입니다. 학습은 *가려진 자리를 맞히는 것*, 생성은 *전부 `[MASK]` 에서 시작해 반복적으로 채우는 것*. Ch 20 의 MLM 과 토큰 수준에서는 똑같이 생겼습니다 — 차이는 *마스킹 비율* 과 *반복 횟수*.

## 3. 토큰화 + `group_texts` (고정 길이 블록 스트림)

Ch 20·24 와 같은 전처리 패턴 — 전체 코퍼스를 토큰화해 이어 붙이고 `block_size=128` 단위로 자릅니다. 특수 토큰 (`[CLS]`, `[SEP]`) 은 넣지 않고 *순수 텍스트 스트림* 으로 만듭니다 (diffusion 은 문장 전체를 한 캔버스로 다루므로 경계 토큰이 불필요).

In [ ]:
BLOCK_SIZE = 128

def tokenize_fn(batch):
    # add_special_tokens=False - [CLS]/[SEP] 없이 순수 토큰 스트림
    return tokenizer(batch["text"], add_special_tokens=False)

tok_train = raw_train.map(tokenize_fn, batched=True, remove_columns=raw_train.column_names, desc="tokenize train")
tok_val   = raw_val.map(tokenize_fn,   batched=True, remove_columns=raw_val.column_names,   desc="tokenize val")

# group_texts - 모든 토큰을 이어붙여 BLOCK_SIZE 단위로 자름
def group_texts(batch):
    concatenated = {k: sum(batch[k], []) for k in batch.keys()}
    total_len = len(concatenated["input_ids"])
    total_len = (total_len // BLOCK_SIZE) * BLOCK_SIZE
    return {
        k: [t[i : i + BLOCK_SIZE] for i in range(0, total_len, BLOCK_SIZE)]
        for k, t in concatenated.items()
    }

lm_train = tok_train.map(group_texts, batched=True, desc="group train")
lm_val   = tok_val.map(group_texts,   batched=True, desc="group val")

# 학습엔 input_ids 만 필요 (마스킹은 collator 가 매번 새로 함)
lm_train = lm_train.remove_columns([c for c in lm_train.column_names if c != "input_ids"])
lm_val   = lm_val.remove_columns([c for c in lm_val.column_names if c != "input_ids"])

print(f"\ntrain chunks: {len(lm_train):,}  (block_size={BLOCK_SIZE})")
print(f"val   chunks: {len(lm_val):,}")
print(f"approx. train tokens: {len(lm_train) * BLOCK_SIZE / 1e6:.2f} M")
print("\nfirst chunk decode (first 200 chars):")
print(tokenizer.decode(lm_train[0]["input_ids"])[:200])

## 4. Diffusion collator — *가변 비율* 마스킹 직접 구현

여기가 BERT MLM 과 갈리는 지점입니다. Ch 20 은 `DataCollatorForLanguageModeling(mlm_probability=0.15)` 로 *고정 15%* 를 가렸지만, diffusion 은 **매 샘플마다 $t \sim U(\epsilon, 1)$ 을 뽑아 그 비율로** 가립니다.

- 각 토큰을 *독립적으로 확률 $t$* 로 `[MASK]` 치환 (LLaDA 의 forward process 와 동일)
- `labels`: 가려진 자리는 원본 토큰 id, 나머지는 `-100` (Ch 20 의 `-100` 트릭 그대로)
- `t`: $1/t$ 재가중을 위해 샘플별 비율도 함께 반환

`add_special_tokens=False` 로 토큰화했으므로 시퀀스 안에 특수 토큰이 없어 *모든 자리가 마스킹 가능* 합니다.

In [ ]:
class DiffusionCollator:
    '''매 배치마다 t ~ U(eps, 1) 을 뽑아 그 비율로 토큰을 [MASK] 치환.'''

    def __init__(self, tokenizer, eps=0.02, seed=42):
        self.mask_id = tokenizer.mask_token_id
        self.eps = eps
        self.gen = torch.Generator().manual_seed(seed)

    def __call__(self, examples):
        ids = torch.tensor([e["input_ids"] for e in examples], dtype=torch.long)
        B, L = ids.shape

        # 샘플별 마스킹 비율 t ~ U(eps, 1)
        t = torch.rand(B, generator=self.gen) * (1.0 - self.eps) + self.eps          # (B,)

        # 각 토큰을 독립적으로 확률 t 로 마스킹
        mask = torch.rand(B, L, generator=self.gen) < t.unsqueeze(1)                  # (B, L) bool
        # 적어도 한 자리는 가리도록 보정 (t 가 아주 작아 전부 안 가려진 경우 방지)
        no_mask_rows = ~mask.any(dim=1)
        if no_mask_rows.any():
            j = torch.randint(0, L, (int(no_mask_rows.sum()),), generator=self.gen)
            mask[no_mask_rows, j] = True

        input_ids = ids.clone()
        input_ids[mask] = self.mask_id
        labels = ids.clone()
        labels[~mask] = -100                                     # 가린 자리만 학습 신호

        attention_mask = torch.ones(B, L, dtype=torch.long)
        return {"input_ids": input_ids, "attention_mask": attention_mask,
                "labels": labels, "t": t}


diff_collator = DiffusionCollator(tokenizer)

# collator 출력 확인 - 같은 두 chunk 를 여러 번 돌리면 매번 다른 비율로 가려짐
print("=== diffusion collator demo (same 2 chunks, masking ratio varies each call) ===")
for trial in range(3):
    batch = diff_collator([lm_train[0], lm_train[1]])
    labels = batch["labels"]
    t = batch["t"]
    for b in range(labels.shape[0]):
        n_masked = (labels[b] != -100).sum().item()
        frac = 100 * n_masked / labels.shape[1]
        print(f"trial {trial} | sample {b}: t={t[b]:.3f}  ->  masked {n_masked:>3d}/{labels.shape[1]} ({frac:5.1f}%)")
    print()

**관전 포인트** — Ch 20 MLM collator 가 *항상 약 15%* 를 가렸다면, 이 collator 는 *호출마다 0-100% 사이 아무 값* 으로 가립니다. 같은 chunk 가 어떤 step 엔 5% 만, 다른 step 엔 90% 가려진 채 학습됩니다 → 모델이 *모든 난이도의 복원* 을 골고루 학습 → 생성 시 *어떤 마스킹 비율에서도* denoise 가능.

> **`-100` thread**: 가려진 자리만 `labels`, 나머지는 `-100`. Ch 20 (MLM 15%) → Ch 28 (SFT, prompt 만 `-100`) → 본 챕터 (가변 마스킹) — 같은 트릭의 세 번째 변주.

## 5. 작은 BERT-style 모델 from scratch

diffusion 의 본체는 *bidirectional encoder* — 가려진 자리를 *좌·우 양방향 문맥* 으로 복원해야 하니 BERT 계열이 자연스럽습니다. `BertForMaskedLM` 을 *random init* 으로 작게 띄웁니다 (Ch 20 의 작은 BERT 와 같은 패턴).

- `num_hidden_layers=4, num_attention_heads=4, hidden_size=256` → 약 3.79M params (작은 vocab 2048 덕분에 임베딩도 가벼움)
- `max_position_embeddings = BLOCK_SIZE = 128`
- MLM head (`Linear(H, V)`) 가 *가려진 자리의 토큰 분포* 를 출력 — 이게 곧 diffusion 의 denoiser

### GPT (Ch 24) 와 코드로 갈리는 곳

- `GPT2LMHeadModel` 이 아니라 `BertForMaskedLM` — *causal mask 없는 bidirectional attention*
- 같은 `from_pretrained` 없이 `BertForMaskedLM(config)` random init — Ch 20·22 와 동일

In [ ]:
from transformers import BertConfig, BertForMaskedLM

config = BertConfig(
    vocab_size=tokenizer.vocab_size,
    hidden_size=256,
    num_hidden_layers=4,
    num_attention_heads=4,
    intermediate_size=1024,
    max_position_embeddings=BLOCK_SIZE,
    pad_token_id=tokenizer.pad_token_id,
)

model = BertForMaskedLM(config).to(device)
n_params = model.num_parameters()
print(f"#params           : {n_params/1e6:.2f} M")
print(f"vocab_size        : {config.vocab_size}")
print(f"\nmodel: {type(model).__name__}")
print(f"  - body : {type(model.bert).__name__}  (Encoder, bidirectional attention)")
print(f"  - head : MLM head -> Linear(in={config.hidden_size}, out={config.vocab_size})")

## 6. Reverse process — 병렬 denoise 생성 함수

diffusion 생성의 핵심. **전부 `[MASK]` 인 시퀀스에서 시작**해 여러 step 에 걸쳐 점점 진짜 토큰으로 채웁니다 (LLaDA 의 *low-confidence remasking* 방식):

1. 현재 `[MASK]` 자리들을 모델이 *한꺼번에* 예측 (병렬!)
2. 각 예측의 *confidence* (softmax 최대 확률) 계산
3. *확신 높은* 자리부터 확정, *확신 낮은* 자리는 다시 `[MASK]` 로 남김
4. 스케줄에 따라 남기는 `[MASK]` 수를 step 마다 줄여 마지막엔 0개

GPT 의 *왼→오 순차* 와 결정적으로 다른 점: **채우는 순서가 위치가 아니라 confidence 순** — 문장 중간이나 끝 단어가 앞 단어보다 먼저 확정될 수 있습니다.

In [ ]:
@torch.no_grad()
def diffusion_generate(active_model, length=64, steps=16, temperature=1.0, top_k=50,
                       prompt_ids=None, record_trajectory=False):
    '''전부 [MASK] 에서 시작해 steps 번 denoise. prompt_ids 를 주면 앞부분 고정 (조건부 생성).

    기본은 sampling (temperature>0). temperature=0 으로 두면 greedy 인데,
    작은 모델 + 전부-[MASK] 출발에서는 greedy 가 최빈 토큰('.')만 뽑는 *붕괴* 가 잘 일어나
    sampling 을 기본값으로 둡니다 (아래 한계 노트 참고).'''
    active_model.eval()
    dev = active_model.device
    mask_id = tokenizer.mask_token_id

    x = torch.full((1, length), mask_id, dtype=torch.long, device=dev)
    fixed = torch.zeros(length, dtype=torch.bool, device=dev)   # 절대 마스킹 안 할 자리 (prompt)
    if prompt_ids is not None:
        p = torch.tensor(prompt_ids[:length], device=dev)
        x[0, :len(p)] = p
        fixed[:len(p)] = True
    n_gen = int((~fixed).sum().item())                          # 생성해야 할 자리 수

    traj = []
    for step in range(steps):
        logits = active_model(input_ids=x).logits[0]            # (L, V)
        probs = logits.softmax(dim=-1)
        if temperature > 0:
            scaled = logits / temperature
            if top_k > 0:                                       # top-k 로 후보 제한
                kth = scaled.topk(top_k, dim=-1).values[:, -1, None]
                scaled = scaled.masked_fill(scaled < kth, float("-inf"))
            pred = torch.multinomial(scaled.softmax(-1), 1).squeeze(-1)
            conf = probs.gather(-1, pred.unsqueeze(-1)).squeeze(-1)
        else:
            conf, pred = probs.max(dim=-1)                      # greedy (최빈 토큰 붕괴 주의)

        is_mask = (x[0] == mask_id) & (~fixed)                  # 지금 마스킹된 (생성 대상) 자리
        # 일단 마스킹된 자리를 예측으로 채운 잠정 시퀀스
        x_new = torch.where(is_mask, pred, x[0])

        # 이 step 이 끝났을 때 남겨둘 [MASK] 수 (선형 스케줄: n_gen -> 0)
        n_remain = int(round(n_gen * (1.0 - (step + 1) / steps)))
        if n_remain > 0:
            # 마스킹됐던 자리들 중 confidence 가 낮은 n_remain 개를 다시 [MASK] 로
            conf_masked = conf.clone()
            conf_masked[~is_mask] = float("inf")               # 마스킹 안 됐던 자리는 후보에서 제외
            remask_idx = conf_masked.topk(n_remain, largest=False).indices
            x_new[remask_idx] = mask_id

        x[0] = x_new
        if record_trajectory:
            traj.append(x[0].clone())

    text = tokenizer.decode(x[0], skip_special_tokens=True)
    return (text, traj) if record_trajectory else text

## 7. 학습 *전* denoise - 비교 기준선 (random init baseline)

학습 전 모델은 가려진 자리를 *균등 추측* 하니, denoise 결과가 *의미 없는 토큰 나열* 이 나옵니다. 학습 후와 나란히 비교하기 위한 기준선 (Ch 20·22 의 *사전학습 전 [MASK] top-5*, Ch 24 의 *학습 전 generation* 과 같은 역할).

In [ ]:
torch.manual_seed(SEED)
print("=" * 70)
print("UNTRAINED model - parallel denoise from all-[MASK]")
print("=" * 70)
for i in range(3):
    text = diffusion_generate(model, length=48, steps=16)
    print(f"\n[sample {i}] {text}")

**관전 포인트** - 학습 전엔 *영어 문장과 거리가 먼 토큰 나열*. logits 가 random 이라 confidence 순서도 무의미. 학습 후 같은 함수로 다시 생성해 비교하면 *diffusion 학습이 본체에 무엇을 새겼는가* 가 드러납니다.

## 8. `Trainer` 로 diffusion 학습 — `1/t` 재가중 loss

BERT/GPT 챕터들과 같은 `Trainer` 패턴이지만, *loss 를 직접 정의* 합니다. `BertForMaskedLM` 의 기본 loss 는 *가려진 자리 CE 평균* 인데, diffusion 은 거기에 *샘플별 `1/t` 재가중* 을 더해야 합니다 (`compute_loss` 오버라이드).

- `DiffusionCollator` → 매 배치 가변 마스킹 + `t` 반환
- `compute_loss` → 가려진 자리 CE 를 샘플별로 합산해 `1/t` 곱한 뒤 평균
- `max_steps=30000`, `batch_size=64`, `fp16=True` - T4 약 19분

In [ ]:
from transformers import Trainer, TrainingArguments, TrainerCallback


class DiffusionTrainer(Trainer):
    '''masked-diffusion loss: 가려진 자리 CE 를 샘플별로 1/t 재가중.'''

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        t = inputs["t"]                                          # (B,)
        labels = inputs["labels"]                               # (B, L)
        outputs = model(input_ids=inputs["input_ids"],
                        attention_mask=inputs["attention_mask"])
        logits = outputs.logits                                 # (B, L, V)
        B, L, V = logits.shape

        per_tok = F.cross_entropy(
            logits.view(-1, V), labels.view(-1),
            ignore_index=-100, reduction="none",
        ).view(B, L)                                            # 가린 자리만 비-0 (나머지 -100 -> 0)

        # 샘플별: (가린 자리 CE 합 / L) * (1/t)
        per_ex = per_tok.sum(dim=1) / L
        loss = (per_ex / t.to(per_ex.dtype)).mean()
        return (loss, outputs) if return_outputs else loss


args = TrainingArguments(
    output_dir="./out_diffusion_intro",
    max_steps=30000,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    fp16=USE_FP16,                       # T4 는 bf16 불가
    logging_steps=50,
    eval_strategy="steps",
    eval_steps=150,
    save_strategy="no",
    report_to="none",
    label_names=["labels"],
    remove_unused_columns=False,         # 'labels','t' 를 collator 가 만들므로 보존
    seed=SEED,
)


class VRAMCallback(TrainerCallback):
    def __init__(self):
        self.steps, self.peak_MiB = [], []

    def on_train_begin(self, args, state, control, **kwargs):
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if torch.cuda.is_available():
            self.steps.append(state.global_step)
            self.peak_MiB.append(torch.cuda.max_memory_allocated() / 1024**2)
            torch.cuda.reset_peak_memory_stats()


vram_cb = VRAMCallback()

trainer = DiffusionTrainer(
    model=model,
    args=args,
    train_dataset=lm_train,
    eval_dataset=lm_val,
    data_collator=diff_collator,
    callbacks=[vram_cb],
)

t0 = time.time()
train_out = trainer.train()
elapsed = time.time() - t0

print(f"\n=== training summary ===")
print(f"elapsed       : {elapsed/60:.2f} min")
print(f"global_step   : {train_out.global_step}")
print(f"train_loss    : {train_out.training_loss:.4f}")
print(f"random baseline (ln vocab): {math.log(tokenizer.vocab_size):.4f}")
if torch.cuda.is_available():
    print(f"final peak    : {torch.cuda.max_memory_allocated()/1024**2:.0f} MiB")

In [ ]:
# loss curve + VRAM trace
log = trainer.state.log_history
train_pts = [(r["step"], r["loss"]) for r in log if "loss" in r and "eval_loss" not in r]
eval_pts  = [(r["step"], r["eval_loss"]) for r in log if "eval_loss" in r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot([s for s, _ in train_pts], [l for _, l in train_pts], "-",
         color="tab:blue", alpha=0.6, label="train")
if eval_pts:
    ax1.plot([s for s, _ in eval_pts], [l for _, l in eval_pts], "s-",
             color="tab:red", label="eval")
ax1.axhline(math.log(tokenizer.vocab_size), ls=":", color="gray",
            label=f"uniform baseline = ln({tokenizer.vocab_size}) approx. {math.log(tokenizer.vocab_size):.2f}")
ax1.set_xlabel("step"); ax1.set_ylabel("diffusion denoising loss (1/t reweighted)")
ax1.set_title("작은 mask-diffusion LM on TinyStories - loss")
ax1.grid(True, alpha=0.3); ax1.legend()

if vram_cb.steps:
    ax2.plot(vram_cb.steps, vram_cb.peak_MiB, "o-", color="tab:green",
             label="peak VRAM (per log window)")
    ax2.set_title(f"VRAM trace  (bs=32, fp16, L={BLOCK_SIZE})")
else:
    ax2.text(0.5, 0.5, "VRAM trace 는 CUDA 에서만 제공",
             ha="center", va="center", transform=ax2.transAxes)
    ax2.set_title("VRAM trace - CUDA 에서만")
ax2.set_xlabel("step"); ax2.set_ylabel("VRAM (MiB)")
ax2.grid(True, alpha=0.3); ax2.legend()

plt.tight_layout(); plt.show()

**관전 포인트** - `1/t` 재가중 덕분에 첫 step loss 가 약 7.6 (`ln(2048)`) 부근에서 시작 (직접 학습한 BPE 2048 의 random baseline 과 같은 값!). 빠르게 떨어져 30000 step 끝에 *약 3.7* 부근에서 안정화되면 정상. 작은 모델 + TinyStories 라 완벽하진 않지만 *가려진 자리를 문맥으로 복원* 하는 능력이 본체에 새겨집니다.

## 9. 학습 *후* denoise + 궤적 시각화

같은 `diffusion_generate` 로 학습 후 생성하고, **denoise 궤적** (각 step 의 시퀀스) 을 출력해 *마스크가 단어로 채워지는 과정* 을 직접 봅니다. 이게 이 챕터의 하이라이트 — GPT 의 왼→오 순차와 달리, *문장 전체가 동시에 흐릿하게 떠오르다 선명해지는* 모습.

In [ ]:
torch.manual_seed(SEED)
print("=" * 70)
print("TRAINED model - parallel denoise from all-[MASK]")
print("=" * 70)
for i in range(3):
    text = diffusion_generate(model, length=48, steps=16)
    print(f"\n[sample {i}] {text}")

In [ ]:
# denoise 궤적 - [MASK] 가 단어로 채워지는 과정을 step 별로
torch.manual_seed(SEED)
text, traj = diffusion_generate(model, length=40, steps=12, record_trajectory=True)

def render(ids):
    toks = tokenizer.convert_ids_to_tokens(ids.tolist())
    return " ".join("____" if tk == tokenizer.mask_token else tk for tk in toks)

print("=" * 78)
print("DENOISE TRAJECTORY  ('____' = still [MASK])  - filled in parallel, by confidence")
print("=" * 78)
n_steps = len(traj)
for step in [0, n_steps // 4, n_steps // 2, 3 * n_steps // 4, n_steps - 1]:
    n_mask = (traj[step] == tokenizer.mask_token_id).sum().item()
    print(f"\nstep {step:>2d}/{n_steps-1}  ([MASK] remaining: {n_mask:>2d})")
    print("  " + render(traj[step]))

print("\n" + "=" * 78)
print("FINAL:", text)

**해석 가이드 - 이게 autoregressive 와 결정적으로 다른 점**

- **step 0**: 거의 전부 `____` (`[MASK]`). 모델이 *가장 확신하는* 몇 자리만 먼저 채워짐 — *위치 순서가 아니라 confidence 순서*. 문장 끝/중간 단어가 앞보다 먼저 나타날 수 있음.
- **중간 step**: 단어들이 *여기저기 동시에* 떠오름. GPT 라면 왼쪽부터 한 칸씩 채워졌을 자리가, diffusion 에선 *전 영역이 함께* 선명해짐.
- **마지막 step**: 모든 `[MASK]` 가 채워진 완성 문장.

> Ch 24 의 GPT generation 이 *왼→오 받아쓰기* 였다면, 여기선 *흐릿한 전체 그림을 반복적으로 다듬기*. 같은 TinyStories 데이터, 같은 "다음 단어가 뭘까" 직관이지만 *생성 메커니즘이 근본적으로 다릅니다.*

## 🔎 솔직한 이야기 — 생성은 되지만 *반복* 이 보인다

학습이 끝난 모델은 전부 `[MASK]` 에서 출발해도 *영어 동화* 를 만들어냅니다 — 인물·대화·배경이 있는 문장이 병렬 denoise 로 채워집니다. 다만 자세히 읽어 보면 **같은 조각이 반복** 되는 게 눈에 띕니다.

> *"Once upon a time, there was a **a** boy named **named** Timmy. ... They are happy friends and happy. They are **to play and play**."*

`named named`, `was a was a`, `play and play` 처럼요. 이건 *모델이 잘못 배운 게 아닙니다.* 고정-$t$(0.15) 복원 정확도가 0.7 안팎까지 오른, 조건부 구조를 제대로 익힌 모델입니다. 반복의 원인은 **샘플러** 에 있습니다.

- 이 챕터의 기본 샘플러는 매 step *confidence 가 높은 자리를 채우고 낮은 자리를 다시 `[MASK]`* 로 두는 방식인데, 한번 "안전한" 고빈도 토큰(`a`, `the`, 자주 나오는 이름)이 높은 confidence 를 받으면 그 토큰이 거듭 뽑히기 쉽습니다.
- 즉 *모델의 확률 분포는 멀쩡한데, 거기서 문장을 어떻게 뽑아내느냐* 가 아직 거친 것입니다.

> 그래서 **다음 Ch 33 은 모델은 그대로 두고 샘플러만 바꿉니다** — carry-over semi-AR + 반복 억제(temperature·top-p·repetition penalty·인접 중복 금지)로 이 반복을 잡아 한결 깔끔한 생성을 얻습니다. 이 챕터에서 "diffusion 이 글을 만든다"를 확인했다면, 다음 챕터는 "그 글을 더 잘 뽑아낸다"입니다.

## 🛠️ 변형 1 - 조건부 생성 (prompt 고정)

조건부 생성은 *문맥(prompt)이 있어* unconditional 보다 잘 동작합니다. 작은 모델의 강점 영역.

GPT 의 prompt 에 대응하는 diffusion 버전: *앞부분 토큰을 고정* (절대 마스킹 안 함) 하고 *나머지만* denoise. "Once upon a time" 을 주고 뒤를 채우게 합니다.

In [ ]:
prompt = "Once upon a time"
prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]

torch.manual_seed(SEED)
print(f"prompt (fixed): {prompt}")
print("=" * 70)
for i in range(3):
    text = diffusion_generate(model, length=48, steps=16, prompt_ids=prompt_ids)
    print(f"\n[sample {i}] {text}")

**관전 포인트** - 앞 토큰들이 고정된 채 뒤가 채워집니다. 단, diffusion 은 *양방향* 이라 GPT 와 달리 *prompt 앞이나 중간에 빈칸* 을 두고 채우게 할 수도 있습니다 (infilling) — autoregressive 가 구조적으로 못 하는 일.

## 🛠️ 변형 2 - denoise step 수 비교 (속도 - 품질 trade-off)

diffusion 만의 자유도: *생성 step 수* 를 바꿀 수 있습니다. GPT 는 토큰 수 = step 수로 고정이지만, diffusion 은 *적은 step (빠르지만 거침) ↔ 많은 step (느리지만 정교)* 을 조절합니다.

In [ ]:
torch.manual_seed(SEED)
for steps in [1, 4, 16, 32]:
    torch.manual_seed(SEED)
    text = diffusion_generate(model, length=48, steps=steps)
    print(f"[steps={steps:>2d}] {text}\n")

**관전 포인트**
- `steps=1` - 전부 `[MASK]` 를 *한 번에* 복원. 문맥 정보가 없어 *서로 안 맞는 단어들* 이 섞이기 쉬움 (각 자리가 독립적으로 예측되니 일관성 ↓).
- `steps=16-32` - 확신 높은 자리부터 단계적으로 확정 → 이미 채운 단어가 *다음 자리의 문맥* 이 되어 일관성 ↑.

> diffusion 생성 품질의 핵심 = *step 수*. 적은 step 은 빠르지만 거칠고, 많은 step 은 느리지만 정교 — 실전 모델 (LLaDA 등) 도 이 trade-off 를 조절합니다.

## ⚖️ Autoregressive (Ch 24) vs Diffusion (본 챕터) 비교

같은 TinyStories, 같은 "언어모델" 이지만 생성 메커니즘이 근본적으로 다릅니다.

| 축 | Autoregressive (GPT, Ch 24) | Diffusion (본 챕터) |
|---|---|---|
| attention | causal (과거만) | **bidirectional (양방향)** |
| 생성 순서 | 왼→오 *위치 순* | **confidence 순 (위치 무관)** |
| 생성 step | 토큰 수 = step (고정) | **임의 (1-32+ 조절)** |
| 병렬성 | 생성 시 순차 (느림) | **여러 자리 동시 생성 (잠재적 고속)** |
| infilling (중간 채우기) | 구조적으로 어려움 | **자연스럽게 가능** (양방향) |
| 출발 상태 | prompt | **전부 `[MASK]`** |
| 성숙도 | 표준 (대부분의 LLM) | **신생 (LLaDA, Trida 등 등장 중)** |

> **왜 diffusion 이 주목받는가**: ① *병렬 생성* 으로 잠재적 속도 이점 (autoregressive 는 토큰 수만큼 순차), ② *양방향 문맥* 으로 infilling·편집에 강점, ③ step 수로 *속도-품질* 을 추론 시점에 조절. 아직 autoregressive 만큼 성숙하진 않지만 *대안 패러다임* 으로 빠르게 발전 중입니다. Ch 33 에서 *사전학습된 작은 diffusion LM (MDLM 170M / DiffuGPT 124M)* 으로 제대로 된 생성을, Ch 34 에서 *한국어 diffusion + AR 직접 비교* 를 다룹니다.

## 📚 이 챕터 알고리즘의 논문 계보

본 챕터에서 *직접 구현* 한 세 요소는 아래 논문들의 방법을 *교육용으로 단순화* 해 옮긴 것입니다. 어느 요소가 어느 논문의 무엇에 대응하는지 정리합니다.

| 구현 요소 (본 챕터) | 대응 논문·수식 | 일치 |
|---|---|---|
| 가변 마스킹 forward (`t ~ U(0,1)`, 토큰별 독립 마스킹) | **LLaDA** Eq. 8 / **D3PM** absorbing-state(=mask) kernel | 동일 |
| `1/t` 재가중 denoising loss (가린 자리 CE 합을 `t·L` 로 정규화) | **LLaDA** Eq. 3 = $-\mathbb{E}[\frac{1}{t}\sum_i \mathbb{1}[x_t^{(i)}{=}\texttt{M}]\log p_\theta]$ / **MDLM** weighted MLM-CE (NELBO) | 동일 |
| low-confidence remasking 생성 (전부 `[MASK]` 시작 → confidence 낮은 자리만 유지) | **LLaDA** sampling (low-confidence remasking) / **MaskGIT** confidence 병렬 디코딩 | 동일 |

> 참고로 LLaDA 논문의 loss 는 본문 수식엔 `1/L` 이 없지만 *구현(Algorithm 1)에서 `t·L` 로 정규화* 합니다. 본 챕터 코드의 `per_tok.sum()/L` 후 `/t` 평균이 정확히 `sum/(t·L)` 으로 *구현 레벨까지 일치* 합니다. 이 loss 는 *negative log-likelihood 의 upper bound* (LLaDA Eq. 4).

### 읽는 순서 추천 (계보)

1. **D3PM** — Austin et al. 2021, [arXiv:2107.03006](https://arxiv.org/abs/2107.03006). 이산 diffusion + *absorbing(=mask) 상태*. 이론 시초.
2. **MaskGIT** — Chang et al. 2022, [arXiv:2202.04200](https://arxiv.org/abs/2202.04200). *confidence 기반 반복 병렬 디코딩* — 본 챕터 생성 절차의 원조 (원래 이미지 분야).
3. **MDLM** — Sahoo et al. 2024, [arXiv:2406.07524](https://arxiv.org/abs/2406.07524). masked diffusion loss = *"고전 MLM loss 들의 가중 혼합"* (NELBO). 본 챕터 `1/t` 재가중의 이론 근거.
4. **LLaDA** — Nie et al. 2025, [arXiv:2502.09992](https://arxiv.org/abs/2502.09992). 위를 *LLM 스케일* 로. **본 챕터가 직접 따른** forward·loss·sampling. 8B 라 Ch 33 의 *대형 맛보기(선택)* 로 다룹니다.

> ⚠️ **혼동 주의** — **Diffusion-LM** (Li et al. 2022, [arXiv:2205.14217](https://arxiv.org/abs/2205.14217)) 은 이름은 비슷하지만 *연속 임베딩 공간* 에서 Gaussian noise 를 더하는 diffusion 이라 본 챕터의 *이산 mask-diffusion* 과 **다른 계열** 입니다. Ch 33 (MDLM/DiffuGPT)·34 는 본 챕터와 같은 이산 mask-diffusion.

> 본 챕터는 *단순화판* 입니다 — 실제 LLaDA 는 semi-autoregressive remasking 등 변형, 대규모 사전학습, 정교한 스케줄을 더합니다. 하지만 *핵심 메커니즘 (가변 마스킹 + `1/t` loss + confidence 병렬 denoise)* 은 동일하므로, 본 챕터를 손으로 구현해 보면 위 논문들의 알고리즘 절을 그대로 읽어낼 수 있습니다.

## 📦 이번 챕터에 등장한 라이브러리·개념

| 이름 | 한 줄 설명 | 다음 챕터에서 |
|---|---|---|
| `BertForMaskedLM(config)` (random init) | bidirectional encoder + MLM head, diffusion 의 denoiser | Ch 33 - MDLM / DiffuGPT (사전학습 diffusion 본체) |
| `DiffusionCollator` (직접 구현) | 매 배치 `t ~ U(0,1)` 가변 마스킹 | Ch 33-34 - 실전 모델은 내부에 동등 로직 |
| `1/t` 재가중 loss (`compute_loss` 오버라이드) | masked-diffusion denoising 목표 (log-likelihood bound) | (개념) LLaDA / MDLM 의 핵심 항 |
| `diffusion_generate` (low-confidence remasking) | 전부 `[MASK]` → 반복 denoise 생성 | Ch 33-34 - 실전 sampler 의 단순화판 |
| `[MASK]` 토큰 (BPE 2048 에 special token 으로 추가, id 2) | forward (가리기) + reverse (생성) 의 캔버스 | Ch 33-34 - 모델별 mask 토큰 |
| denoise 궤적 시각화 | 마스크 → 단어 병렬 채움 관찰 | (개념) AR 과의 핵심 대비 |

## 🎯 체크포인트 질문

1. BERT MLM (Ch 20) 의 *고정 15% 마스킹* 과 diffusion 의 *가변 마스킹* 은 collator 코드에서 정확히 무엇이 다른가요? 왜 diffusion 은 비율을 가변으로 둬야 *생성* 이 가능할까요?
2. diffusion loss 의 `1/t` 재가중이 없으면 어떤 일이 생길까요? (힌트: `t=0.05` 인 샘플과 `t=0.95` 인 샘플의 loss 크기 비교)
3. `diffusion_generate` 에서 *왜 confidence 낮은 자리를 다시 `[MASK]` 로 남기는가* 를 설명해 보세요. 한 번에 다 확정하면 (`steps=1`) 왜 품질이 떨어질까요?
4. autoregressive (GPT) 가 구조적으로 못 하는 *infilling (문장 중간 빈칸 채우기)* 을 diffusion 은 왜 자연스럽게 할 수 있나요? (causal vs bidirectional attention 관점)

## ❓ FAQ

### Q1. (이론) diffusion LM 은 결국 BERT MLM 과 뭐가 다른가요? 같은 거 아닌가요?

**메커니즘은 거의 같고, *목적과 사용법* 이 다릅니다.** BERT MLM 은 *고정 15% 를 한 번 가려 복원* 하며 *표현* 을 배우는 게 목적 (이후 downstream fine-tune). Diffusion LM 은 *가변 0-100% 마스킹 + 반복 denoise* 로 *생성* 그 자체가 목적입니다.

핵심 일반화 두 가지:
- **마스킹 비율 일반화**: 15% (고정) → $t \sim U(0,1)$ (가변). 100% 가린 상태까지 학습했기에 *전부 `[MASK]` 에서 출발하는 생성* 이 가능.
- **반복 적용**: MLM 은 1회 복원, diffusion 은 *여러 step* 에 걸쳐 점진적 복원.

```python
# BERT MLM (Ch 20) - 고정 비율, 1회
DataCollatorForLanguageModeling(tokenizer, mlm=True, mlm_probability=0.15)

# Diffusion (본 챕터) - 가변 비율 + 1/t 재가중, 생성 시 반복 denoise
t = torch.rand(B) * (1 - eps) + eps           # 매번 다른 비율
mask = torch.rand(B, L) < t.unsqueeze(1)
```

즉 *BERT 를 이미 안다면 diffusion LM 의 80% 를 이미 아는 셈* 입니다.

### Q2. (이론) `1/t` 재가중은 왜 필요한가요?

**마스킹 비율에 무관하게 loss 척도를 맞추고, 학습 목표가 *log-likelihood 의 upper bound* 가 되게 하기 위함** 입니다.

재가중이 없으면: `t=0.05` 샘플은 가려진 토큰이 약 6개뿐이라 CE 합이 작고, `t=0.95` 샘플은 약 122개라 CE 합이 큽니다. 그대로 평균하면 *많이 가린 샘플이 loss 를 지배* → 학습이 *어려운 (거의 다 가린) 경우에만* 편향됩니다.

`1/t` 를 곱하면 (수식상 가린 토큰 수가 평균적으로 $tL$ 이므로) *모든 t 의 기여가 비슷해져* 균형이 맞고, 동시에 이 형태가 *연속시간 diffusion 의 변분 하한 (ELBO)* 과 일치합니다 (LLaDA / MDLM 의 유도). 본 챕터에서 첫 step loss 가 *어떤 t 든 `ln(vocab)` 으로 정렬* 되는 게 그 증거.

### Q3. (실무) 생성 결과가 GPT (Ch 24) 보다 거친데 정상인가요?

**정상이고, 작은 from-scratch diffusion 의 *구조적* 한계입니다.** 두 가지를 구분하세요.

1. **greedy 붕괴** — 전부 `[MASK]` 에서 greedy(argmax) 로 뽑으면 문맥 없는 첫 step 에서 최빈 토큰(`.`)이 모든 자리 최고 confidence 라 *마침표만 반복* 됩니다. 그래서 `diffusion_generate` 의 기본은 sampling (`temperature=1.0, top_k=50`). greedy 는 진단·비교용으로만.
2. **규모 한계** — sampling 으로 바꿔도 작은 모델의 unconditional 생성은 거칩니다. 이건 *알고리즘이 아니라 규모* 문제예요: 같은 작은 규모에서 *표준 BERT MLM(고정 15%) 도 복원이 비슷하게 약하고*, `1/t` 재가중 유무도 차이가 없습니다. loss 가 `ln(vocab)` 에서 잘 내려간 것 자체가 학습은 정상이라는 뜻.

품질을 올리려면 규모를 키우거나(아래) — 더 현실적으로는 *사전학습된 작은 모델* 을 쓰면 됩니다 (Ch 33).

```python
# 규모 키우기 (T4 30분 안에서 가능한 선)
args.max_steps = 3000
config.num_hidden_layers = 6; config.hidden_size = 384
diffusion_generate(model, length=64, steps=32)  # 생성 step 도 늘리기
```

*제대로 된 diffusion 생성* 은 Ch 33 에서 — **MDLM (170M) / DiffuGPT (124M)** 같은 사전학습 모델이 *같은 알고리즘, 충분한 규모* 로 얼마나 달라지는지 직접 봅니다.

### Q4. (실무) `steps` 를 늘리면 무조건 좋아지나요?

**어느 지점까지는 좋아지고, 그 뒤로는 포화** 됩니다. 적은 step (`steps=1`) 은 모든 자리를 독립적으로 한 번에 확정해 *서로 안 맞는 단어* 가 섞이기 쉽고, step 을 늘리면 *이미 확정한 단어가 다음 자리의 문맥* 이 되어 일관성이 오릅니다. 하지만 step 수가 시퀀스 길이를 넘어가면 *더 줄일 `[MASK]` 가 없어* 이득이 사라집니다.

trade-off: `steps` ↑ → 품질 ↑, 속도 ↓. 실전에선 *길이의 절반 정도* 가 흔한 출발점 (예: length=64 → steps=32). diffusion 의 매력은 *이 값을 추론 시점에 자유롭게* 정할 수 있다는 것 — autoregressive 는 불가능.

### Q5. (이론) diffusion 이 autoregressive 보다 빠를 수 있다는데 왜 본 챕터는 안 빨라 보이나요?

**잠재적 병렬성** 때문입니다. autoregressive 는 토큰 N 개 생성에 *반드시 N 번 순차* forward (이전 토큰이 있어야 다음을 생성). diffusion 은 *step 수만큼만* forward 하면 되고 (step < N 가능), 각 step 에서 *여러 자리를 동시에* 채웁니다.

본 챕터에서 안 빨라 보이는 이유: 작은 모델 + 짧은 시퀀스라 forward 1회가 워낙 빨라 *오버헤드가 묻힘*. 긴 시퀀스 + 큰 모델 + 최적화된 sampler 에서 이점이 드러납니다. 다만 *현재 실전 성숙도* 는 autoregressive 가 여전히 앞섭니다 (KV-cache 등 최적화 누적). diffusion 은 *발전 중인 대안*.

### Q6. (실무) BPE 2048 을 직접 학습하면서 `[MASK]` 토큰은 어떻게 마련했나요?

**BPE 를 학습할 때 `special_tokens` 로 함께 등록** 했습니다. diffusion 의 forward/reverse 모두 `[MASK]` 가 핵심인데, 일반 BPE/WordPiece 어휘에는 `[MASK]` 가 없을 수 있습니다. 이 챕터는 작은 from-scratch 모델에 맞춰 *vocab 을 작게* 가져가려고 `bert-base-uncased` 의 WordPiece(30,522) 를 그대로 쓰지 않고, TinyStories 코퍼스에 ByteLevel BPE 를 vocab 2,048 으로 직접 학습합니다. 이때 `BpeTrainer(special_tokens=["[PAD]", "[UNK]", "[MASK]"])` 로 세 특수 토큰을 어휘 맨 앞에 고정 배정해 `[MASK]` 가 id 2 에 자리 잡습니다.

```python
trainer = trainers.BpeTrainer(vocab_size=2048,
                              special_tokens=["[PAD]", "[UNK]", "[MASK]"])
# 학습 후: tokenizer.mask_token_id == 2
```

모델은 `BertForMaskedLM` 을 *random init* 으로 띄우므로 임베딩도 처음부터 함께 학습됩니다 — `[MASK]` 임베딩이 별도 부담이 아니라 본체와 같이 자라납니다. bidirectional encoder (`BertForMaskedLM`) 와 `[MASK]` 기반 denoising 이 자연스럽게 짝을 이룹니다.

### Q7. (이론) 그럼 앞으로 autoregressive 는 사라지나요?

**가까운 미래엔 아닙니다.** autoregressive 는 *성숙도 (KV-cache, 방대한 인프라·최적화), 안정적 품질, 검증된 스케일링* 에서 여전히 표준입니다. diffusion LM 은 *병렬 생성·infilling·step 조절* 이라는 차별점으로 *특정 용도* (빠른 생성, 편집, 제약 만족) 에서 주목받는 *대안* 입니다.

둘은 *대체* 라기보다 *공존·융합* 으로 가는 중 (일부 연구는 둘을 섞음). 본 커리큘럼이 *둘 다 직접 구현* (Ch 24 GPT, Ch 32 diffusion) 해 본 이유 — *생성 패러다임의 지형* 을 손으로 익혀 두면 어느 쪽이 발전하든 따라갈 수 있습니다.

## 다음 챕터 예고

**Chapter 33. 작은 사전학습 Diffusion LM — MDLM (170M) + DiffuGPT (124M) 추론**

- **MDLM-owt** (`kuleshov-group/mdlm-owt`, 170M, arXiv:2406.07524) — 본 챕터가 직접 따른 *바로 그 masked diffusion 논문* 의 공식 체크포인트. `AutoModelForMaskedLM` (fill-mask) 라 본 챕터 `BertForMaskedLM` 과 *인터페이스가 거의 동일* → 코드가 매끄럽게 이어집니다. T4 여유.
- **DiffuGPT-small** (`diffusionfamily/diffugpt-s`, 124M, arXiv:2410.17891) — *가장 작은* 정식 사전학습 diffusion LM. GPT2 본체라 **Ch 24 (GPT, autoregressive) 와 같은 본체에서 AR vs diffusion 직접 비교** 가능.
- 본 챕터 작은 from-scratch 모델과 *품질 격차* 를 직접 체감 — *같은 알고리즘, 충분한 규모* 면 unconditional 생성이 얼마나 달라지는지.
- (대형 맛보기) LLaDA-8B 는 4bit 양자화로 *선택 실습*.

> **변하는 축**: *모델 출발점* (scratch 약 3.79M → 사전학습 170M / 124M). 메커니즘 (병렬 denoise) 은 본 챕터에서 이미 손으로 구현해 봤습니다. Ch 34 에서 *한국어 diffusion + autoregressive 직접 비교* 로 Phase 5 를 마무리합니다.